# Sparse R-CNN on synthetic shapes: run on a Colab GPU

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. The full run trains 2 configurations × 3 seeds (up to 80 epochs each, early stopping on validation mAP) and evaluates the selected checkpoints on a 1,000-image held-out test set. Each epoch prints its duration.
3. The last cell downloads `results.zip`. Unzip it into the repo folder (it contains a `results/` folder).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Upload `sparse-rcnn-shapes.zip` when prompted. Once the repo is on GitHub, set `REPO_URL` to clone it instead.

In [ ]:
import os, zipfile
REPO_URL = None   # e.g. 'https://github.com/<user>/sparse-rcnn-shapes.git'
REPO_DIR = 'sparse-rcnn-shapes'
if not os.path.exists(REPO_DIR):
    if REPO_URL:
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        from google.colab import files
        uploaded = files.upload()
        zipfile.ZipFile(next(iter(uploaded))).extractall('.')
%cd {REPO_DIR}
!pip install -q torchmetrics pycocotools

## 2. (Optional) Save results to Google Drive as they finish
`metrics.json` is written after each configuration, so a disconnect costs at most the configuration still training. Rerun it with `--only <name>`; results merge.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    target = '/content/drive/MyDrive/sparse-rcnn-shapes-results'
    os.makedirs(target, exist_ok=True)
    if os.path.isdir('results') and not os.path.islink('results'):
        !cp -rn results/. {target}/ && rm -rf results
    if not os.path.exists('results'):
        os.symlink(target, 'results')

## 3. Smoke test (about a minute; writes to `results/smoke/`, which is git-ignored)

In [ ]:
!python -u src/run_experiments.py --smoke

## 4. Full run
Default: `sparse_rcnn` and `single_scale`, seeds 42/43/44. Optional extra: `--only more_data` (5x training images, roughly 5x slower per epoch).

In [ ]:
!mkdir -p results && python -u src/run_experiments.py 2>&1 | tee results/run_log.txt

## 5. (Optional) More training data

In [ ]:
# !python -u src/run_experiments.py --only more_data 2>&1 | tee results/run_log_more_data.txt

## 6. Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/results_table.md', encoding='utf-8').read()))
for f in ['training_curves', 'map_per_stage', 'sparse_rcnn_predictions', 'sparse_rcnn_proposals']:
    display(Image(f'results/figures/{f}.png'))

## 7. Download (checkpoints included, smoke test excluded)

In [ ]:
!zip -qr results.zip results -x 'results/smoke/*'
from google.colab import files
files.download('results.zip')